In [1]:
from __future__ import print_function,division
from keras.datasets import cifar10
from keras.layers import BatchNormalization,Activation,Dense,Input,Reshape,Flatten,Dropout,multiply,GaussianNoise,Embedding,ZeroPadding2D,MaxPooling2D,LeakyReLU,UpSampling2D,Conv2D
from keras.optimizers import Adam
from keras.models import Sequential,Model
from keras import losses
from keras.utils import to_categorical
import keras.backend as k
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class Encoder():
    def __init__(self):
        self.rows=32
        self.cols=32
        self.mask_h=8
        self.mask_w=8
        self.channels=3
        self.classes=2
        self.shape=(self.rows,self.cols,self.channels)
        self.missing_shape=(self.mask_h,self.mask_w,self.channels)
        opt=Adam(learning_rate=0.0002,beta_1=0.5)
        self.discriminator=self.build_discriminator()
        self.discriminator.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])
        self.generator=self.build_generator()
        masked_img=Input(shape=self.shape)
        gen_missing=self.generator(masked_img)
        self.discriminator.trainable=False
        valid=self.discriminator(gen_missing)
        self.combined=Model(masked_img,[gen_missing,valid])
        self.combined.compile(loss=['mse','binary_crossentropy'],loss_weights=[0.999,0.001],optimizer=opt)

    def build_generator(self):
        model=Sequential()
        model.add(Input(shape=self.shape))
        model.add(Conv2D(32,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Conv2D(64,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Conv2D(128,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Conv2D(512,kernel_size=1,strides=2,padding='same'))
        model.add(Dropout(0.5))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(UpSampling2D())
        model.add(Conv2D(128,kernel_size=3,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation('relu'))
        model.add(UpSampling2D())
        model.add(Conv2D(64,kernel_size=3,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(Activation('relu'))
        model.add(Conv2D(self.channels,kernel_size=3,padding='same'))
        model.add(Activation('tanh'))
        model.summary()
        masked_img=Input(shape=self.shape)
        gen_missing=model(masked_img)
        return Model(masked_img,gen_missing)

    def build_discriminator(self):
        model=Sequential()
        model.add(Input(shape=self.missing_shape))
        model.add(Conv2D(64,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Conv2D(128,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Conv2D(256,kernel_size=3,strides=2,padding='same'))
        model.add(BatchNormalization(momentum=0.8))
        model.add(LeakyReLU(negative_slope=0.2))
        model.add(Flatten())
        model.add(Dense(1,activation='sigmoid'))
        model.summary()
        img=Input(shape=self.missing_shape)
        validity=model(img)
        return Model(img,validity)

    def mask_data(self,imgs):
        y1=np.random.randint(0,self.rows-self.mask_h+1,imgs.shape[0])
        y2=y1+self.mask_h
        x1=np.random.randint(0,self.cols-self.mask_w+1,imgs.shape[0])
        x2=x1+self.mask_w
        masked_imgs=np.empty_like(imgs)
        missing_parts=np.empty((imgs.shape[0],self.mask_h,self.mask_w,self.channels))
        for i,img in enumerate(imgs):
            masked_img=img.copy()
            _y1,_y2,_x1,_x2=y1[i],y2[i],x1[i],x2[i]
            missing_parts[i]=masked_img[_y1:_y2,_x1:_x2,:].copy()
            masked_img[_y1:_y2,_x1:_x2,:]=0
            masked_imgs[i]=masked_img
        return masked_imgs,missing_parts,(y1,y2,x1,x2)

    def train(self,epochs,batch_size=128,sample_interval=50):
        (x_train,y_train),(_,_)=cifar10.load_data()
        x_cats=x_train[(y_train==3).flatten()]
        x_dogs=x_train[(y_train==5).flatten()]
        x_train=np.vstack((x_cats,x_dogs))
        x_train=x_train/127.5-1.
        valid=np.ones((batch_size,1))
        fake=np.zeros((batch_size,1))
        for epoch in range(epochs):
            idx=np.random.randint(0,x_train.shape[0],batch_size)
            imgs=x_train[idx]
            masked_imgs,missing_parts,_=self.mask_data(imgs)
            gen_missing=self.generator.predict(masked_imgs,verbose=0)
            d_loss_real=self.discriminator.train_on_batch(missing_parts,valid)
            d_loss_fake=self.discriminator.train_on_batch(gen_missing,fake)
            d_loss=0.5*np.add(d_loss_real,d_loss_fake)
            g_loss=self.combined.train_on_batch(masked_imgs,[missing_parts,valid])
            print("%d [D loss: %f, acc: %.2f%%] [G loss: %f, mse: %f]"%(epoch,d_loss[0],100*d_loss[1],g_loss[0],g_loss[1]))
            if epoch%sample_interval==0:
                idx=np.random.randint(0,x_train.shape[0],6)
                imgs=x_train[idx]
                self.sample_images(epoch,imgs)
            if epoch==24900:
                self.save_model()

    def sample_images(self,epoch,imgs):
        r,c=3,6
        masked_imgs,missing_parts,(y1,y2,x1,x2)=self.mask_data(imgs)
        gen_missing=self.generator.predict(masked_imgs,verbose=0)
        imgs=0.5*imgs+0.5
        masked_imgs=0.5*masked_imgs+0.5
        gen_missing=0.5*gen_missing+0.5
        fig,axis=plt.subplots(r,c)
        for i in range(c):
            axis[0,i].imshow(imgs[i,:,:])
            axis[0,i].axis('off')
            axis[1,i].imshow(masked_imgs[i,:,:])
            axis[1,i].axis('off')
            filled_img=imgs[i].copy()
            filled_img[y1[i]:y2[i],x1[i]:x2[i],:]=gen_missing[i]
            axis[2,i].imshow(filled_img)
            axis[2,i].axis('off')
        plt.savefig('images/%d.png'%epoch)
        plt.close()

    def save_model(self):
        def save(model,model_name):
            model_path="saved_model%s.json"%model_name
            weights_path="saved_model%s.weights.h5"%model_name
            options={"file_arch":model_path,"file_weight":weights_path}
            json_string=model.to_json()
            open(options["file_arch"],"w").write(json_string)
            model.save_weights(options["file_weight"])
        save(self.generator,"generator")
        save(self.discriminator,"discriminator")

In [3]:
!mkdir images

In [4]:
!mkdir saved_model

In [5]:
if __name__=="__main__":
  context_encoder=Encoder()
  context_encoder.train(epochs=25000,batch_size=64,sample_interval=1000)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 4, 4, 64)       │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 4, 4, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 4, 4, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 2, 2, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 2, 2, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 1, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 1, 1, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 372,865 (1.42 MB)

 Trainable params: 371,969 (1.42 MB)

 Non-trainable params: 896 (3.50 KB)

...

 Trainable params: 825,603 (3.15 MB)

 Non-trainable params: 832 (3.25 KB)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1792s 11us/step


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/trainer.py:86: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


Streaming output truncated to the last 5000 lines.
20000 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124578, mse: 0.124002]
20001 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124576, mse: 0.124001]
20002 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124575, mse: 0.123999]
20003 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124574, mse: 0.123998]
20004 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124572, mse: 0.123997]
20005 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124571, mse: 0.123996]
20006 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124569, mse: 0.123994]
20007 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124568, mse: 0.123993]
20008 [D loss: 0.694414, acc: 53.23%] [G loss: 0.124567, mse: 0.123992]
20009 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124566, mse: 0.123991]
20010 [D loss: 0.694415, acc: 53.23%] [G loss: 0.124566, mse: 0.123991]
20011 [D loss: 0.694414, acc: 53.23%] [G loss: 0.124564, mse: 0.123989]
20012 [D loss: 0.694414, acc: 53.23%] [G loss: 0.124563, mse: 0.123988]
20013 [D loss

In [6]:
import os
os.makedirs("inpainting",exist_ok=True)
generator_json=context_encoder.generator.to_json()
with open("inpainting/generator.json","w") as json_file:
  json_file.write(generator_json)
context_encoder.generator.save_weights("inpainting/generator.weights.h5")
discriminator_json=context_encoder.discriminator.to_json()
with open("inpainting/discriminator.json","w") as json_file:
  json_file.write(discriminator_json)
context_encoder.discriminator.save_weights("inpainting/discriminator.weights.h5")

In [7]:
import shutil
shutil.make_archive("inpainting","zip",".","inpainting")
from google.colab import files
files.download("inpainting.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>